In [22]:
from math import sqrt
import random
import src.generate_encodings as ge
import src.prediction_models as pm
import src.predictor_optimizer as pop
from src.metrics import *
from copy import copy
import warnings
from src.utils import HiddenPrints, HiddenWarnings
import os, sys
from tqdm import tqdm
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import numpy as np
import gc
import time

In [23]:
start_time = time.time()

In [24]:
debugging = False
demo_case = False
update_params_after_each_cycle = True
skip_initial_hyperparameter_tuning = True

In [25]:
"""defining Parameters for Input_Data"""

#sequence params
dataset = "GFP"
activity_threshold = 1  #Threshold for considering a mutants score as active or inactive for all datasets (data hab been preprocessed)
repr_type = 'blosum80' #blosum metrics, OHE, Georgiev, ESM
score_column = "Norm_score_1"  # if true all data ranges from 0 to 1, with the activity threshold differs for every dataset


"""Defining further ModelParameter"""

#mlde params
r_top = 0.99  #Percentage cutoff of top scoring datapoints, targeted to be identified during the MLDE CyclesS87F:Q95K:K112T:L198P
n_gain = 50  # realistic range of obtained samples per iteration: [10,20,50,100] -> sys arg
n_starting_points = 1000  #realistic, up to [100, 200, 500, 1000, 2000, 5000] points to be expected as common practice -> sys arg
n_attempts = 50  # Number of training attempts per cycle to build a better performing model than for the previous cylcle. 
r_start = 0.9  # Decimal of mutants with the highest ddG to select starting points from
n_cycles = 60  # Since one cycle is expected to take at least 1 month (lab validation), I do not expect the whole project to take for more than 5 years.
delta_excl = 0.5  #Threshold to exclude the mutants with Zero-Shot score above

#model params
model_type = "xgboost"  # xgboost, rf, lightgbm, adaboost, svr, linear, ridge, lasso, svr
cv_folds = 5
early_stopping_fraction = 0.01

#hypertuning params
initial_trials = 100 if model_type not in ["linear", "ridge",
                                          "lasso"] else 10000  #trials per group to optimize the parameters for the mlde model - at the very beginning before the benchmark.
n_trials = 50  # trials per group to optimize the parameters for the mlde model - after each cycle.
target_metric = "spearman"  # "spearman", "ndcg", "pearson", "mse", "mae", "r2"

benchmark_run = 1 # run number of the benchmark, to be used for the file name, 20 in total
benchmark_ID = f"{dataset}_{repr_type}_{model_type}_nGain{n_gain}_nStart{n_starting_points}_rStart{int(r_start * 100)}_nCycles{n_cycles}_normedScores_{score_column}_run{benchmark_run}"

# Generate timestamp in format DDMMYYYY_HH_MM
from datetime import datetime
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M") 

In [26]:
# Setting up output directories
out = f'../Results/MLDE_Benchmark/{benchmark_ID}_{timestamp}'
if not os.path.exists(out):
    os.makedirs(out)

# Create log file
log_file = f'{out}/00_Benchmark_log.txt'
if not os.path.exists(log_file):
    with open(log_file, 'w') as log:
        log.write(f"MLDE Benchmark Log\n")
        log.write(f"Benchmark ID: {benchmark_ID}\n")
        log.write(f"Timestamp: {timestamp}\n")
        log.write(f"\n")

In [ ]:
#prepare Dataset and Embeddings

if repr_type not in ["esmc_600m", "esmc300m"]:
    import_data = f"../Data/Protein_Gym_Datasets/{dataset}.csv"
    data = []  # id, sequence, embedding (x), score (y),zs_score ddG (z)
    headers = []

    with open(import_data, "r") as infile:
        for i, line in enumerate(infile.readlines()[:]):
            line = line[:-1].split(",")
            if i == 0:
                headers = line
                continue

            id = line[0]
            sequence = line[1]
            embedding = ge.generate_sequence_encodings(repr_type, [sequence])[0]
            label = round(float(line[headers.index(score_column)]), 3)
            zs_score = round(float(line[6]), 3)
            data.append((id, sequence, embedding, label, zs_score))

else:  #load referring ESM Embeddings
    print("Loading ESM Embeddings not implemented yet.")

#Determining Parameters for MLDE Benchmark
wild_type = data[0][1]
mutations = data[0][0].split(":")
for mut in mutations:
    index = int(mut[1:-1]) - 1
    wild_type = wild_type[:index] + mut[0] + wild_type[index + 1:]
max_score = max([isxyz[3] for isxyz in data])
max_starting_score = round(max([isxyz[3] for isxyz in data]) * r_start, 3)
target_score = round(max_score * r_top, 3)

#Define the library of potential mutants by filtering out those with a ΔΔG above the threshold
library= [isxyz for isxyz in data if isxyz[4] <= delta_excl]

with open(log_file, 'a') as log:
    log.write(f"============================ Dataset-Parameters ============================\n")
    log.write(f"Applied Data-Set: {dataset}\n")
    log.write(f"Number of datapoints: {len(data)}\n")
    log.write(f"Wildtype Sequence: {wild_type}\n")
    log.write(f"Library-Size (=Number of mutants within threshold): {len(library)}\n")
    log.write(f"ΔΔG-Threshold for Exclusion: >{delta_excl} kcal/mol\n")
    log.write(f"maximum score: {max([isxyz[3] for isxyz in data])}\n")
    log.write(f"minimum score: {min([isxyz[3] for isxyz in data])}\n")
    log.write(f"\n")
    log.write(f"============================ Model-Parameters ============================\n")
    log.write(f"Protein Representation Type: {repr_type}\n")
    log.write(f"Algorithm: {model_type}\n")
    log.write(f"Cross Validation Folds: {cv_folds}\n")
    log.write(f"Early Stopping Fraction (if applicable): {early_stopping_fraction}\n")
    log.write(f"\n")
    log.write(f"============================ MLDE-Parameters ============================\n")
    log.write(f"Number of Cycles: {n_cycles}\n")
    log.write(f"Number of Starting Points: {n_starting_points}\n")
    log.write(f"Number of Gain Points per Cycle: {n_gain}\n")
    log.write(f"Top ΔΔG-Percentage for Starting Points Selection: {1-(int(r_start*100))}%\n")
    log.write(f"Target Score to achieve ({int(r_top*100)}% of Top-Score): {target_score}\n")
    log.write(f"Number of Training Attempts per Cycle: {n_attempts}\n")
    log.write(f"\n")
    log.write(f"============================ Hyper-Parameter-Tuning-Parameters ============================\n")
    log.write(f"n Trials for initial Hyper-Parameter Tuning: {initial_trials}\n")
    log.write(f"n Trials for Hyper-Parameter Tuning after each Cycle: {n_trials}\n")
    log.write(f"n Parameter to optimize for: {target_metric}\n")
    log.write(f"\n")

In [ ]:
#Function to display chosen datapoints

# Assuming y_mlde, remaining_data, data, and data_set_name_shortened are already defined

def display_datapoints_distribution(y_mlde, remaining_data, iteration: int = None, r_top: float = False,
                                    show_in_browser: bool = False):
    min_score = round(float(min([float(isxyz[3]) for isxyz in data])), 3)
    max_score = round(float(max([float(isxyz[3]) for isxyz in data])), 3)

    n_bins = 100
    plotwidth = 2000

    binned_mlde_scores = dict()
    binned_remaining_data = dict()

    bin_edges = np.linspace(min_score, max_score, n_bins)

    def bin_value(y, bin_edges):
        lower = float(10 ** -9)
        upper = float(10 ** 9)
        y = float(y)
        for i in bin_edges:
            if i < y:
                lower = float(i)
            else:
                upper = float(i)
                break
        return lower if abs(y - lower) < abs(y - upper) else upper

    # First loop for y_mlde
    for y in y_mlde:
        binned = bin_value(y, bin_edges)
        binned_mlde_scores[binned] = binned_mlde_scores.get(binned, 0) + 1

    # Second loop for remaining_data
    for isxy in remaining_data:
        y = isxy[3]
        binned = bin_value(y, bin_edges)
        binned_remaining_data[binned] = binned_remaining_data.get(binned, 0) + 1

    # Create ONE subplot only
    scoring_plt = make_subplots(rows=1, cols=1)

    # Add shapes to the plot
    shapes = []
    shape_names = ["Allowed Starting Range", "Targeted Scoring Range"]
    target_score = r_top * max_score if r_top else None
    shape_colors = ["rgba(178, 34, 34, 0.2)", "rgba(32, 85, 5, 0.2)"]
    shape_xs = [[0, max_starting_score], [target_score, max_score]]

    for i in [0, 1] if r_top else [1]:
        shapes.append(
            dict(
                type="rect",
                xref="x", yref="paper",
                x0=shape_xs[i][0], x1=shape_xs[i][1],
                y0=0, y1=1,
                fillcolor=shape_colors[i],
                line=dict(width=0),
                layer="below"
            )
        )

        scoring_plt.add_trace(
            go.Scatter(
                x=[None], y=[None],
                mode='markers',
                marker=dict(size=0),
                fill='toself',
                fillcolor=shape_colors[i],
                name=shape_names[i],
                showlegend=True
            ),
            row=1, col=1
        )

    # Add grey bars first (background)
    scoring_plt.add_trace(
        go.Bar(
            name="Remaining-Data-Points",
            x=list(binned_remaining_data.keys()),
            y=list(binned_remaining_data.values()),
            marker=dict(color="grey")
        ),
        row=1, col=1
    )

    # Add red bars on top (foreground)
    scoring_plt.add_trace(
        go.Bar(
            name=f"Known Points ({sum(binned_mlde_scores.values())})",
            x=list(binned_mlde_scores.keys()),
            y=list(binned_mlde_scores.values()),
            marker=dict(color="red")
        ),
        row=1, col=1
    )

    if iteration != None:
        title = f"Discovered- vs To-Discover-Datapoints (binned) from {dataset}-Dataset used within {iteration}. MLDE Cycle"
    else:
        title = f"Known Starting- vs To-Discover-Datapoints (binned) from {dataset}-Dataset used within MLDE Cycle"

    scoring_plt.update_layout(
        title_text=title,
        title_font=dict(color="black", size=20),
        showlegend=True,
        barmode='overlay',  # Important for overlapping bars
        paper_bgcolor='rgb(233,233,233)',
        plot_bgcolor='rgb(233,233,233)',
        shapes=shapes,
        width=plotwidth,
        height=plotwidth * 0.6,
        legend=dict(
            font=dict(color="black", size=12)

        )
    )

    scoring_plt.update_yaxes(
        dict(
            type="log",
            title_text="Number of sequences",
            title_font=dict(color="black"),
            color='black',
            showgrid=True,
            linecolor='black',
            gridcolor='grey',
            griddash="dot",
            gridwidth=0.1
        )
    )

    scoring_plt.update_xaxes(
        dict(
            title_text="Binned Activity Score of sequence",
            title_font=dict(color="black"),
            range=[min_score, max_score],
            color='black',
            linecolor='black',
            showgrid=True,
            gridcolor='grey',
            griddash="dot",
            gridwidth=0.1
        )
    )
    import plotly.io as pio
    if show_in_browser:
        pio.renderers.default = "browser"
    else:
        pio.renderers.default = "notebook_connected"
    scoring_plt.show()
    return scoring_plt

In [ ]:
#Selecting Datapoints for Benchmark
try:
    mlde_datapoints = random.sample(sorted(library, key=lambda isxyz: isxyz[4])[:int(r_start*len(library))], n_starting_points)
except ValueError as e:
    with open(log_file, 'a') as log:
        log.write("[ERROR] Amount of available, allowed Amount of Active or Inactive Starting Points does not meet Criteria of max_quote_of_inactives and highest_starting_fraction!")
    raise ValueError(
        "Amount of available, allowed Amount of Active or Inactive Starting Points does not meet Criteria of max_quote_of_inactives and highest_starting_fraction!")


sequences_mlde = [isxyz[1] for isxyz in mlde_datapoints]
x_mlde = [isxyz[2] for isxyz in mlde_datapoints]
y_mlde = [isxyz[3] for isxyz in mlde_datapoints]

remaining_data = [isxyz for isxyz in library if isxyz not in mlde_datapoints]
random.shuffle(remaining_data)
target_score = r_top * max_score

with open(log_file, 'a') as log:
    log.write("[INFO] Declared MLDE Starting-Points and remaining dataset (Details: 01_Starting_Points)\n")

In [ ]:
starting_plot = display_datapoints_distribution(y_mlde=y_mlde, remaining_data=remaining_data, r_top=r_top,
                                                show_in_browser=False)
starting_plot.write_image(f"{out}/01_Starting_Points.png")

In [ ]:
"""Create a suitable hyperparameter-set for the initial MLDE-Model"""

if skip_initial_hyperparameter_tuning:
    with open(log_file, 'a') as log:
        log.write("[INFO] Skipping initial Hyper-Parameter Tuning for MLDE Model.\n")
        log.write("\n")

    mlde_params = {'early_stopping_rounds': 1, 'subsample': 0.970093388927376, 'colsample_bytree': 0.8116904750007311, 'max_depth': 64, 'min_child_weight': 1.4415229752899685, 'learning_rate': 0.19538445839567264, 'n_estimators': 188}

else:
    mlde_optimizer = pop.Sequential_Optimizer(model_type=model_type, cv_folds=cv_folds, x_arr=x_mlde, y_arr=y_mlde,
                                            initial_params={},
                                            trials_per_group=initial_trials,
                                            early_stopping_fraction=early_stopping_fraction,
                                            n_jobs=1)

    mlde_optimizer.optimize_stepwise()
    best_trial = mlde_optimizer.get_best_trial()
    mlde_params = mlde_optimizer.get_best_params()
    with open(log_file, 'a') as log:
        log.write(f"[INFO] Initial Hyper-Parameter Tuning for {model_type} model completed.\n")
        log.write("\n")

In [ ]:
gc.collect()  # clear memory

93

In [ ]:
finished = False

with open(log_file, 'a') as log:
    log.write(f"============================ MLDE-Benchmark-Start ============================\n")
    log.write("\n")
    log.write(f"Starting the MLDE-Benchmark with {n_cycles} cycles, starting at {n_starting_points} sequences. \n")
    log.write(f"Discovery of {n_gain} per Cycle to identify the highest scoring sequences >= {target_score} (i.e. {r_top}%) of max from list.\n")
    log.write(f"Initial Hyper-Parameters for {model_type} model:\n{mlde_params}\n")
    log.write("\n")

scored_mutants = []  # save each iterations highest achieved sequence score and the average over all sequences and standard deviation
train_performances = []  # Performance for each cycle's trained model on the training data
val_performances = []  # Performance for each cycle's trained model on the validation data
test_performances = []  # Performance for each cycle's tested model on the remaining data
ood_performances = []  # Performance for each cycle's tested model the potential targets (mostly out of Distribution)

previous_Spearman = float(-1)
current_Spearman = float(-1)
best_cycle_Spearman = float(-1)

for i in range(1, n_cycles + 1):
    print(f"########################## Starting Cycle {i}/{n_cycles} ##########################")
    print()

    with open(log_file, 'a') as log:
        log.write(f"########################## Starting Cycle {i}/{n_cycles} ##########################\n")
        log.write("\n")

    best_cycle_model = None
    j = 0

    with tqdm(total=n_attempts,
              desc=f'Training a model {n_attempts} times to obtain the better performing model') as pbar:

        while j < n_attempts:
            mlde_model = pm.ActivityPredictor(model_type=model_type,
                                            x_arr=x_mlde,
                                            y_arr=y_mlde,
                                            shuffle_data=True,
                                            early_stopping=10,
                                            params=mlde_params)
            with HiddenPrints():
                with HiddenWarnings():
                    mlde_model.train(k_folds=cv_folds)
                    current_Spearman = round(mlde_model.get_performance()[0], 3)
                    current_Pearson = mlde_model.get_performance()[1]

            

            if current_Pearson is float('nan') or current_Spearman is float('nan'):
                continue # skip this iteration if Pearson or Spearman is NaN, which can happen with some models

            j += 1
            pbar.update(1)

            if current_Spearman > best_cycle_Spearman or best_cycle_model is None:
                best_cycle_Spearman = copy(current_Spearman)
                best_cycle_model = copy(mlde_model)

    
    #obtain Trainingsperformance
    y_train_pred = best_cycle_model.predict(best_cycle_model.get_data(prepared=True)["x_train"])
    y_train = best_cycle_model.get_data(prepared=True)["y_train"]

    train_NDCG = round(ndcg_score([y for y in y_train_pred], [y for y in y_train]), 3)
    train_Spearman = round(spearman_correlation([y for y in y_train_pred], [y for y in y_train]), 3)
    train_Pearson = round(pearson_correlation([y for y in y_train_pred], [y for y in y_train]), 3)
    train_R2 = round(r2_score([y for y in y_train_pred], [y for y in y_train]), 3)
    train_MSE = round(mse([y for y in y_train_pred], [y for y in y_train]), 3)

    val_NDCG = round(best_cycle_model.get_performance()[0], 3)
    val_Spearman = round(best_cycle_model.get_performance()[1], 3)
    val_Pearson = round(best_cycle_model.get_performance()[2], 3)
    val_R2 = round(best_cycle_model.get_performance()[3], 3)
    val_MSE = round(best_cycle_model.get_performance()[4], 3)


    with open(log_file, 'a') as log:
        log.write(
            f"Best attempt's Val-Performance after {j} training attempts: \n"
            f"NDCG {val_NDCG}, (Spearman: {val_Spearman}, Pearson: {val_Pearson}, R2: {val_R2}, MSE: {val_MSE})\n"
            f"Training Performance for the best Attempt: \n"
            f"NDCG {train_NDCG}, (Spearman: {train_Spearman}, Pearson: {train_Pearson}, R2: {train_R2}, MSE: {train_MSE})\n"
        )
        log.write("\n")

    # Predict on the remaining data 
    list_predictions = []
    with tqdm(total=len(remaining_data), desc="Predicting all remaining datapoints within the dataset...") as pbar:
        with HiddenPrints():
            with HiddenWarnings():
                for isxy in remaining_data:
                    list_predictions.append(best_cycle_model.predict([isxy[2]])[0])
                    pbar.update(1)

    test_NDCG = round(ndcg_score([y for y in list_predictions], [isxy[3] for isxy in remaining_data]), 3)
    test_Spearman = round(spearman_correlation([y for y in list_predictions], [isxy[3] for isxy in remaining_data]), 3)
    test_Pearson = round(pearson_correlation([y for y in list_predictions], [isxy[3] for isxy in remaining_data]), 3)
    test_R2 = round(r2_score([y for y in list_predictions], [isxy[3] for isxy in remaining_data]), 3)
    test_MSE = round(mse([y for y in list_predictions], [isxy[3] for isxy in remaining_data]), 3)

    with open(log_file, 'a') as log:
        log.write(f"Interference-Performance on all remaining datapoints (incl. Out of Distribution Prediction):\n"
            f"NDCG {test_NDCG}, (Spearman: {test_Spearman}, Pearson: {test_Pearson}, R2: {test_R2}, MSE: {test_MSE})\n")
        log.write("\n")

    top_predictions = sorted([(isxy, y_head) for isxy, y_head in zip(remaining_data, list_predictions)],
                             key=lambda tuple: tuple[1], reverse=True)[:n_gain]


    top_predictions = sorted(top_predictions, key=lambda tuple: tuple[0][3], reverse=True)

    with open(log_file, 'a') as log:
        log.write(f'Top {50} identified Samples:\n')
        for target in top_predictions:
            if target[0][3] < target_score:
                log.write(f'{target[0][0]}, Predicted: {round(float(target[1]), 3)}, True: {target[0][3]}\n')
            if target[0][3] >= target_score:
                log.write(f'{target[0][0]}, Predicted: {round(float(target[1]), 3)}, True: {target[0][3]} - Top Mutant identified!\n')
                finished = True
        log.write("\n")

    ood_NDCG = round(ndcg_score([isxy[0][3] for isxy in top_predictions], [y[1] for y in top_predictions]), 3)
    ood_Spearman = round(
        spearman_correlation([isxy[0][3] for isxy in top_predictions], [y[1] for y in top_predictions]), 3)
    ood_Pearson = round(pearson_correlation([isxy[0][3] for isxy in top_predictions], [y[1] for y in top_predictions]),
                        3)
    ood_R2 = round(r2_score([isxy[0][3] for isxy in top_predictions], [y[1] for y in top_predictions]), 3)
    ood_MSE = round(mse([isxy[0][3] for isxy in top_predictions], [y[1] for y in top_predictions]), 3)

    with open(log_file, 'a') as log:
        log.write(f"Interference-Performance on selected target datapoints (incl. Out of Distribution Prediction):\n"
            f"NDCG {ood_NDCG}, (Spearman: {ood_Spearman}, Pearson: {ood_Pearson}, R2: {ood_R2}, MSE: {ood_MSE})\n")
        log.write("\n")

    scored_mutants.append(top_predictions)
    train_performances.append((train_NDCG, train_Spearman, train_Pearson, train_R2, train_MSE))
    val_performances.append((val_NDCG, val_Spearman, val_Pearson, val_R2, val_MSE))
    test_performances.append((test_NDCG, test_Spearman, test_Pearson, test_R2, test_MSE))
    ood_performances.append((ood_NDCG, ood_Spearman, ood_Pearson, ood_R2, ood_MSE))

    if finished:
        break

    #else:
    #update mlde trainingpoints-range
    for target in top_predictions:
        x_mlde.append(target[0][2])
        y_mlde.append(target[0][3])

    #shuffle the mlde_data (in fact not necessary, but better save than sorry)
    mlde_data = [(x, y) for x, y in zip(x_mlde, y_mlde)]
    random.shuffle(mlde_data)
    x_mlde = [xy[0] for xy in mlde_data]
    y_mlde = [xy[1] for xy in mlde_data]

    #update remaining data
    remaining_data = [isxy for isxy in remaining_data if isxy not in [isxy_yhead[0] for isxy_yhead in top_predictions]]
    random.shuffle(remaining_data)

    highest_score = max([target[0][3] for target in top_predictions])
    mean_score = round((sum([target[0][3] for target in top_predictions]) / n_gain), 3)
    standard_dev = round(
        sqrt(sum([(y - mean_score) ** 2 for y in [target[0][3] for target in top_predictions]]) / n_gain), 3)



    #prepare next cycle:
    # ->update last iterations previous_Spearman
    previous_Spearman = copy(best_cycle_Spearman)

    params_updated = False
    if update_params_after_each_cycle:
        mlde_optimizer = pop.Sequential_Optimizer(model_type=model_type, cv_folds=cv_folds, x_arr=x_mlde, y_arr=y_mlde,
                                                  initial_params=copy(mlde_params),
                                                  trials_per_group=int(n_trials),
                                                  early_stopping_fraction=early_stopping_fraction,
                                                  n_jobs=1)

        with HiddenPrints():
            mlde_optimizer.optimize_stepwise() # the optimizer will compare current hyperparams with with "altered params" and will only overrite them in case of improvement
            best_trial = mlde_optimizer.get_best_trial()

        if mlde_params != mlde_optimizer.get_best_params():
            mlde_params = mlde_optimizer.get_best_params()  # might even overwrite with the same params
            params_updated = True
        else:
            params_updated = False

    with open(log_file, 'a') as log:
        log.write(("[INFO] Hyperparameters updated.\n") if params_updated else "[INFO] Current Hyperparameters maintained.\n")
        log.write("\n")

    print('hyperparameters updated.' if params_updated else 'current hyperparameters maintained.')
with open(log_file, 'a') as log:
    log.write(
        f"MLDE-Performance-Ranking finished after {i}/{n_cycles} cycles {"successfully" if finished else "without success"}.")

print(f"MLDE-Performance-Ranking finished after {i}/{n_cycles} cycles {'successfully' if finished else 'without success'}.")

########################## Starting Cycle 1/60 ##########################



Training a model 50 times to obtain the better performing model: 100%|██████████| 50/50 [01:29<00:00,  1.80s/it]
Predicting all remaining datapoints within the dataset...: 100%|██████████| 22117/22117 [00:24<00:00, 917.87it/s]
[I 2025-07-16 12:11:09,620] A new study created in RDB with name: MLDE-Model


  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

...

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

hyperparameters updated.
########################## Starting Cycle 5/60 ##########################



Training a model 50 times to obtain the better performing model: 100%|██████████| 50/50 [02:47<00:00,  3.35s/it]
Predicting all remaining datapoints within the dataset...: 100%|██████████| 21917/21917 [00:24<00:00, 889.12it/s]


MLDE-Performance-Ranking finished after 5/60 cycles successfully.


In [ ]:
"""Saving The Performance and Scored Mutants"""
files = ["03_Training_Performances.txt", "04_Validation_Performances.txt", "05_Test_Performances.txt", "06_Scored_Mutants.txt"]

for i, data in enumerate([train_performances, val_performances, test_performances, scored_mutants]):
    with open(f"{out}/{files[i]}", "w") as f:
        f.write(f"MLDE-Benchmark Performance Data {files[i][:-4]} for {benchmark_ID} - {timestamp}\n")
        for entry in data:
            f.write(f"{entry}\n")

In [ ]:
"""Display Results: Development of Predictions and Mutant-Selection"""

import plotly.graph_objects as go
from plotly.subplots import make_subplots

results_plot = make_subplots(
    subplot_titles=["Development of True Mutant Scores for Targets","Performance on all remaining datapoints",
    "Performance on Training Set","Performance on Validation Set"],
    rows=2, cols=2)

'''[1,1] Plot for Results Tracking'''
results_plot.append_trace(
    go.Scatter(name="Highest Scoring Mutant", x=list(range(1,len(scored_mutants)+1)),
               y=[max([target[0][3] for target in targets]) for targets in scored_mutants],
               marker=dict(color="rgba(230, 20, 20, 0.8)", size=3),
               mode="lines"), row=1, col=1)

results_plot.append_trace(
    go.Scatter(name="Average of Mutants' Score", x=list(range(1,len(scored_mutants)+1)),
               y=[np.mean([target[0][3] for target in targets]) for targets in scored_mutants],
               marker=dict(color="rgba(255, 0, 255, 0.8)", size=3),
               mode="lines"), row=1, col=1)

results_plot.append_trace(
    go.Scatter(name="Standard Deviation of Mutant Scores", x=list(range(1,len(scored_mutants)+1)),
               y=[np.std([target[0][3] for target in targets],ddof=1) for targets in scored_mutants],
               marker=dict(color="rgba(128, 0, 128, 0.8)", size=3),
               mode="lines"), row=1, col=1)

'''[1,2] Plot for Performance Tracking against all remaining datapoints'''

results_plot.append_trace(
    go.Scatter(name="NDCG", x=list(range(1,len(test_performances)+1)),
               y=[metric[0] for metric in test_performances],
               marker=dict(color="rgba(0, 0, 0, 0.8)", size=3),
               mode="lines"), row=1, col=2)

results_plot.append_trace(
    go.Scatter(name="Spearman Ranking Correlation Coefficient", x=list(range(1,len(test_performances)+1)),
               y=[metric[1] for metric in test_performances],
               marker=dict(color="rgba(255, 215, 0, 0.8)", size=3),
               mode="lines"), row=1, col=2)

results_plot.append_trace(
    go.Scatter(name="Pearson Correlation Coefficient", x=list(range(1,len(test_performances)+1)),
               y=[metric[2] for metric in test_performances],
               marker=dict(color="rgba(34, 139, 34, 0.8)", size=3),
               mode="lines"), row=1, col=2)

results_plot.append_trace(
    go.Scatter(name="R2", x=list(range(1,len(test_performances)+1)),
               y=[metric[3] for metric in test_performances],
               marker=dict(color="rgba(0, 128, 128, 0.8)", size=3),
               mode="lines"), row=1, col=2)

results_plot.append_trace(
    go.Scatter(name="MSE", x=list(range(1,len(test_performances)+1)),
               y=[metric[4] for metric in test_performances],
               marker=dict(color="rgba(190, 40, 0, 0.8)", size=3),
               mode="lines"), row=1, col=2)

'''[2,1] Plot for Performance Tracking against Validation set'''

results_plot.append_trace(
    go.Scatter(name="NDCG", x=list(range(1,len(train_performances)+1)),
               y=[metric[0] for metric in train_performances],
               marker=dict(color="rgba(0, 0, 0, 0.8)", size=3),
               mode="lines"), row=2, col=1)

results_plot.append_trace(
    go.Scatter(name="Spearman Ranking Correlation Coefficient", x=list(range(1,len(train_performances)+1)),
               y=[metric[1] for metric in train_performances],
               marker=dict(color="rgba(255, 215, 0, 0.8)", size=3),
               showlegend=False,
               mode="lines"), row=2, col=1)

results_plot.append_trace(
    go.Scatter(name="Pearson Correlation Coefficient", x=list(range(1,len(train_performances)+1)),
               y=[metric[2] for metric in train_performances],
               marker=dict(color="rgba(34, 139, 34, 0.8)", size=3),
               showlegend=False,
               mode="lines"), row=2, col=1)

results_plot.append_trace(
    go.Scatter(name="R2", x=list(range(1,len(train_performances)+1)),
               y=[metric[3] for metric in train_performances],
               marker=dict(color="rgba(0, 128, 128, 0.8)", size=3),
               showlegend=False,
               mode="lines"), row=2, col=1)

results_plot.append_trace(
    go.Scatter(name="MSE", x=list(range(1,len(train_performances)+1)),
               y=[metric[4] for metric in train_performances],
               marker=dict(color="rgba(190, 40, 0, 0.8)", size=3),
               showlegend=False,
               mode="lines"), row=2, col=1)

'''[2,2] Plot for Performance Tracking against Validation set'''

results_plot.append_trace(
    go.Scatter(name="NDCG", x=list(range(1,len(val_performances)+1)),
               y=[metric[0] for metric in val_performances],
               marker=dict(color="rgba(0, 0, 0, 0.8)", size=3),
               mode="lines"), row=2, col=2)

results_plot.append_trace(
    go.Scatter(name="Spearman Ranking Correlation Coefficient", x=list(range(1,len(val_performances)+1)),
               y=[metric[1] for metric in val_performances],
               marker=dict(color="rgba(255, 215, 0, 0.8)", size=3),
               showlegend=False,
               mode="lines"), row=2, col=2)

results_plot.append_trace(
    go.Scatter(name="Pearson Correlation Coefficient", x=list(range(1,len(val_performances)+1)),
               y=[metric[2] for metric in val_performances],
               marker=dict(color="rgba(34, 139, 34, 0.8)", size=3),
               showlegend=False,
               mode="lines"), row=2, col=2)

results_plot.append_trace(
    go.Scatter(name="R2", x=list(range(1,len(val_performances)+1)),
               y=[metric[3] for metric in val_performances],
               marker=dict(color="rgba(0, 128, 128, 0.8)", size=3),
               showlegend=False,
               mode="lines"), row=2, col=2)

results_plot.append_trace(
    go.Scatter(name="MSE", x=list(range(1,len(val_performances)+1)),
               y=[metric[4] for metric in val_performances],
               marker=dict(color="rgba(190, 40, 0, 0.8)", size=3),
               showlegend=False,
               mode="lines"), row=2, col=2)



results_plot.update_layout(
    title_text=f"Development of Predictions and Mutant-Selection to identify the {r_top} active sequences testing {n_gain} new samples per iteration with {model_type}-{repr_type} for {dataset}",
    title_font=dict(color="black", size=20),
    showlegend=True,
    paper_bgcolor='rgb(233,233,233)',
    plot_bgcolor='rgb(233,233,233)',
    height=1000,
    width=1500,
    legend=dict(font=dict(color="black",
                          size=12)))
#

results_plot.update_yaxes(
    dict(
        title_text="Activity-Score",
        title_font=dict(color="black"),
        # range=[min_score, max_score],
        color='black',
        showgrid=True,
        gridcolor='grey',
        linecolor='black',
        griddash="dot",
        gridwidth=0.2,
        dtick=0.25))

results_plot.update_xaxes(
    dict(
        title_text="cycles",
        title_font=dict(color="black"),
        # range=[min_score, max_score],
        color='black',
        showgrid=True,
        gridcolor='grey',
        linecolor='black',
        griddash="dot",
        gridwidth=0.2,
        dtick=1))

import plotly.io as pio

pio.renderers.default = "notebook_connected"
results_plot.show()

results_plot.write_image(f"{out}/02_Results.jpg")

In [ ]:
finishing_time = time.time() - start_time

# Convert to hours:minutes:seconds format
hours = int(finishing_time // 3600)
minutes = int((finishing_time % 3600) // 60)
seconds = int(finishing_time % 60)
finishing_time_formatted = f"{hours:02d}:{minutes:02d}:{seconds:02d}"

print(f"Total execution time: {finishing_time_formatted}")
finishing_time_minutes = round(finishing_time / 60, 2)  # keep original minutes for logging

Total execution time: 02:06:39
